# Librerias

In [1]:
import sys
import os

# Agrega la carpeta raíz del proyecto al path de Python
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nbformat
import openpyxl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from keras.metrics import Precision, Recall, AUC
import pickle

from src.Data_Processor import DataProcessor

# Carga de datos

In [2]:
procesador = DataProcessor().load_data('../data/processed/Proyecto_B_Clientes_Limpio.parquet')

In [3]:
procesador.data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Adulto_Mayor                  7043 non-null   int64  
 1   Pareja                        7043 non-null   int64  
 2   Dependientes                  7043 non-null   int64  
 3   Antiguedad(Meses)             7043 non-null   float64
 4   Servicio_Telefono             7043 non-null   int64  
 5   Lineas_Multiples              7043 non-null   int64  
 6   Internet_DSL                  7043 non-null   int8   
 7   Internet_Fiber optic          7043 non-null   int8   
 8   Seguridad_Online              7043 non-null   int64  
 9   Copia_Seguridad_Online        7043 non-null   int64  
 10  Proteccion_Dispositivos       7043 non-null   int64  
 11  Soporte_Tecnico               7043 non-null   int64  
 12  Streaming_TV                  7043 non-null   int64  
 13  Streaming_Peli

# Particion de datos

Extraccion de datos para validación

In [4]:
val_churn_data = procesador.data[procesador.data['Abandono'] == 1].sample(n=9, axis=0, random_state=42)
val_non_churn_data = procesador.data[procesador.data['Abandono'] == 0].sample(n=14, axis=0, random_state=42)

procesador.data.drop(val_churn_data.index, inplace=True)
procesador.data.drop(val_non_churn_data.index, inplace=True)

In [5]:
procesador.data.shape

(7020, 21)

In [6]:
val_churn_data.head(10)

,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Internet_DSL,Internet_Fiber optic,Seguridad_Online,Copia_Seguridad_Online,...,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,MP_Electronic check,MP_Mailed check,MP_Bank transfer (automatic),Cargos_Mensuales,Abandono
6444,0,0,1,31.0,1,1,1,0,0,1,...,1,1,1,1,1,0,1,0,79.45,1
5634,0,1,1,1.0,1,0,0,0,0,0,...,0,0,0,0,1,1,0,0,19.90,1
3528,0,0,0,16.0,1,0,0,0,0,0,...,0,0,0,0,1,0,1,0,19.70,1
1057,0,0,0,10.0,1,0,0,1,1,0,...,0,1,0,0,1,0,1,0,84.70,1
1579,1,1,0,12.0,1,0,0,1,0,0,...,0,0,1,0,1,1,0,0,80.45,1
6257,1,0,0,1.0,1,0,0,1,0,1,...,0,0,0,0,1,1,0,0,73.00,1
973,1,1,0,44.0,0,0,1,0,0,1,...,0,0,0,0,1,1,0,0,30.35,1
1458,0,0,0,1.0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,20.55,1
1236,0,0,0,3.0,1,1,0,1,0,0,...,0,0,0,0,1,1,0,0,76.95,1


In [7]:
val_non_churn_data.head(15)

,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Internet_DSL,Internet_Fiber optic,Seguridad_Online,Copia_Seguridad_Online,...,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,MP_Electronic check,MP_Mailed check,MP_Bank transfer (automatic),Cargos_Mensuales,Abandono
2134,0,0,0,1.0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,20.15,0
2693,0,0,0,72.0,1,0,0,0,0,0,...,0,0,0,2,0,0,0,1,19.30,0
1679,1,0,1,7.0,1,0,1,0,0,0,...,0,0,0,0,1,0,1,0,45.75,0
5998,0,1,0,48.0,1,0,0,0,0,0,...,0,0,0,2,1,0,1,0,20.05,0
6798,0,0,0,13.0,1,0,0,1,0,0,...,0,0,0,0,1,0,1,0,70.15,0
1994,0,0,0,22.0,1,0,0,0,0,0,...,0,0,0,1,0,0,1,0,19.60,0
1531,0,1,1,35.0,1,1,0,0,0,0,...,0,0,0,0,1,1,0,0,25.75,0
5089,0,1,1,8.0,1,0,1,0,0,0,...,0,1,0,0,1,1,0,0,54.40,0
6560,0,0,1,23.0,1,0,0,0,0,0,...,0,0,0,1,0,0,1,0,19.80,0
707,1,0,0,27.0,1,1,0,1,0,1,...,1,1,1,1,1,0,0,1,104.65,0


In [8]:
validation_set = pd.concat([val_churn_data, val_non_churn_data], axis=0)

In [9]:
validation_set

,Adulto_Mayor,Pareja,Dependientes,Antiguedad(Meses),Servicio_Telefono,Lineas_Multiples,Internet_DSL,Internet_Fiber optic,Seguridad_Online,Copia_Seguridad_Online,...,Soporte_Tecnico,Streaming_TV,Streaming_Peliculas,Contrato,Facturacion_Electronica,MP_Electronic check,MP_Mailed check,MP_Bank transfer (automatic),Cargos_Mensuales,Abandono
6444,0,0,1,31.0,1,1,1,0,0,1,...,1,1,1,1,1,0,1,0,79.45,1
5634,0,1,1,1.0,1,0,0,0,0,0,...,0,0,0,0,1,1,0,0,19.90,1
3528,0,0,0,16.0,1,0,0,0,0,0,...,0,0,0,0,1,0,1,0,19.70,1
1057,0,0,0,10.0,1,0,0,1,1,0,...,0,1,0,0,1,0,1,0,84.70,1
1579,1,1,0,12.0,1,0,0,1,0,0,...,0,0,1,0,1,1,0,0,80.45,1
6257,1,0,0,1.0,1,0,0,1,0,1,...,0,0,0,0,1,1,0,0,73.00,1
973,1,1,0,44.0,0,0,1,0,0,1,...,0,0,0,0,1,1,0,0,30.35,1
1458,0,0,0,1.0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,20.55,1
1236,0,0,0,3.0,1,1,0,1,0,0,...,0,0,0,0,1,1,0,0,76.95,1
2134,0,0,0,1.0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,20.15,0


Se formo un subconjunto pequeño de datos para manejar datos con la misma estructur del dataset para corroborar las predicciones con una estructura que el modelo no conozca solo para racilitar la ejemplificacion de las predcciones.

## Muestreo estratificado

In [10]:
# Se hace la partición, se remueven los datos automáticamente del set de entrenamiento
# y se mantiene la proporción perfecta de la columna 'Abandono' gracias a 'stratify'
x = procesador.data.drop('Abandono', axis=1)
y = procesador.data['Abandono']

X_train, X_test, y_train, y_test = train_test_split(
    x, 
    y, 
    test_size=0.15,        # Porcentaje para prueba 
    stratify=y, # para mantener el balanceo de los datos en la columna 'Abandono'
    random_state=42
)

In [11]:
X_train.shape

(5967, 20)

In [12]:
X_test.shape

(1053, 20)

In [13]:
y_train.shape

(5967,)

In [14]:
y_test.shape

(1053,)

# Escalado

In [15]:
scaler = MinMaxScaler()

In [16]:
X_train_scaled= scaler.fit_transform(X_train)
X_test_scaled= scaler.transform(X_test)

In [17]:
X_train_scaled


array([[0.        , 1.        , 1.        , ..., 0.        , 0.        ,
        0.91334661],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.06523904],
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.16683267],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.87051793],
       [0.        , 1.        , 0.        , ..., 0.        , 0.        ,
        0.48605578],
       [0.        , 1.        , 0.        , ..., 1.        , 0.        ,
        0.15438247]], shape=(5967, 20))

In [18]:
type(X_train_scaled)


numpy.ndarray

In [19]:
type(X_test_scaled)


numpy.ndarray

In [20]:
type(y_train)


pandas.Series

In [26]:
type(y_test)


pandas.Series

In [22]:
type(validation_set)

pandas.DataFrame

# Guardar scaler y particiones

In [23]:

# Guardar el scaler en un archivo .pkl
with open('../models/scaler.pkl', 'wb') as archivo:
    pickle.dump(scaler, archivo)
    


In [ ]:
# Guardar subsets
subsets = [X_train_scaled, X_test_scaled, y_train, y_test, validation_set]

paths = ['../data/partitions/X_train_scaled.npy',
         '../data/partitions/X_test_scaled.npy',
         '../data/partitions/y_train.csv',
         '../data/partitions/y_test.csv',
         '../data/partitions/validation_set.parquet']

for subset, path in zip(subsets, paths):
    if path.endswith('.npy'):
        procesador.array = subset
    else:
        procesador.data = subset
    procesador.save_data(path, index=False)
    
